# Projeto Integrador III — Pipeline de dados industriais no Microsoft Fabric

Este notebook lê as leituras das máquinas no **MySQL (Clever Cloud)**, cria todas as tabelas
da solução e grava tudo no **Lakehouse**, seguindo a arquitetura Medalhão:

| Camada | Tabelas | Para que serve |
|---|---|---|
| 🥉 **Bronze** | `bronze_leituras` | Cópia fiel do banco, sem alterações. Só recebe dados novos. |
| 🥈 **Prata** | `prata_torno`, `prata_prensa`, `prata_esteira`, `prata_robo`, `prata_compressor`, `prata_embaladora`, `prata_leituras`, `prata_medidas`, `prata_quarentena` | Dados validados, uma tabela por máquina, com os deltas dos contadores. |
| 🥇 **Ouro** | `dim_maquina`, `dim_indicador`, `config_limites`, `fato_classificacao`, `fato_ocorrencias`, `fato_paradas`, `fato_indicadores_periodo`, `fato_estado_atual` | O que o painel (Power BI) usa: alertas, indicadores e estado atual. |

## Antes de rodar
1. Crie um **Lakehouse** chamado `lh_industria` no workspace (sem a opção de *schemas*).
2. No painel **Explorer** à esquerda do notebook, adicione esse Lakehouse e deixe-o como **padrão**.
3. Configure a **senha do MySQL** fora do código (veja a célula de configuração).
4. Rode todas as células (**Run all**). Na primeira vez, o notebook importa todo o histórico.
5. Depois, agende o notebook (**Run → Schedule**) para rodar a cada poucos minutos.
   Cada execução importa só as leituras novas e recalcula as camadas Prata e Ouro.

In [ ]:
# Biblioteca para conectar ao MySQL (o Fabric não traz instalada)
%pip install mysql-connector-python --quiet

In [ ]:
# ================================================================
# 1. CONFIGURAÇÃO — o único lugar que normalmente precisa ser alterado
# ================================================================
import json, os
from datetime import datetime, timezone
from functools import reduce
from operator import or_
from zoneinfo import ZoneInfo

# --- Banco de origem (MySQL no Clever Cloud) ---
MYSQL_HOST    = "bems2qe8mmsk3n0cc6vx-mysql.services.clever-cloud.com"
MYSQL_PORTA   = 3306
MYSQL_BANCO   = "bems2qe8mmsk3n0cc6vx"
MYSQL_USUARIO = "ugxdzs5tdh0voozx"
MYSQL_TABELA  = "leituras"
FUSO_DO_BANCO = "UTC"   # fuso dos horários gravados no MySQL

# --- Senha: NUNCA escreva a senha aqui. Use UMA das opções abaixo ---
# Opção A (recomendada): Azure Key Vault
KEY_VAULT_URL     = ""              # ex.: "https://kv-pi3.vault.azure.net/"
KEY_VAULT_SEGREDO = "mysql-senha"
# Opção B: arquivo no Lakehouse, em Files/config/mysql.json, com o conteúdo {"senha": "..."}
ARQUIVO_SENHA = "/lakehouse/default/Files/config/mysql.json"

# --- Lakehouse de destino ---
LAKEHOUSE = "lh_industria"
FORMATO   = "delta"

# --- Parâmetros de negócio ---
JANELA_INDICADORES       = "15 minutes"  # tamanho do período em fato_indicadores_periodo
LIMITE_SEM_COMUNICACAO_S = 15            # regra do descritivo
INTERVALO_MAXIMO_S       = 60            # intervalos maiores não contam como tempo de operação

In [ ]:
# ================================================================
# 2. CONEXÃO COM O LAKEHOUSE
# ================================================================
spark.conf.set("spark.sql.session.timeZone", "UTC")

try:
    spark.sql(f"USE {LAKEHOUSE}")
except Exception as erro:
    raise RuntimeError(
        f"Não encontrei o Lakehouse '{LAKEHOUSE}'. No painel Explorer do notebook, "
        "adicione o Lakehouse e defina-o como padrão. Depois rode esta célula de novo."
    ) from erro

print("✔ Conectado ao Lakehouse:", spark.catalog.currentDatabase())

In [ ]:
# ================================================================
# 3. CATÁLOGO DAS MÁQUINAS E REGRAS DE NEGÓCIO
# ================================================================
# Cada campo: (unidade, natureza, confirmado)
#   natureza "instantaneo" = valor medido no momento (temperatura, vibração...)
#            "acumulado"   = contador desde que a ESP32 ligou (precisa de delta)
#            "nivel"       = cresce aos poucos, mas é comparado direto com o limite (desgaste)
#   confirmado True  = campo já observado nos dados reais (vira obrigatório na validação)
#              False = previsto pelo descritivo; confirmar o nome quando a máquina enviar dados
MAQUINAS = {
    "TORNO_CNC_01": {
        "nome": "Torno CNC", "setor": "Usinagem", "ordem_fluxo": 1, "tabela": "prata_torno",
        "campos": {
            "temperatura": ("°C", "instantaneo", True),
            "vibracao":    ("mm/s", "instantaneo", True),
            "rotacao":     ("RPM", "instantaneo", True),
            "desgaste":    ("%", "nivel", True),
            "producao":    ("peças", "acumulado", True),
            "defeitos":    ("peças", "acumulado", True),
        }},
    "PRENSA_HIDRAULICA_01": {
        "nome": "Prensa hidráulica", "setor": "Conformação", "ordem_fluxo": 2, "tabela": "prata_prensa",
        "campos": {
            "pressao":     ("bar", "instantaneo", False),
            "temperatura": ("°C", "instantaneo", False),
            "vibracao":    ("mm/s", "instantaneo", False),
            "ciclos":      ("ciclos", "acumulado", False),
            "defeitos":    ("peças", "acumulado", False),
        }},
    "ROBO_SOLDAGEM_01": {
        "nome": "Robô de soldagem", "setor": "Soldagem", "ordem_fluxo": 3, "tabela": "prata_robo",
        "campos": {
            "temperatura":    ("°C", "instantaneo", False),
            "corrente":       ("A", "instantaneo", False),
            "ciclos":         ("ciclos", "acumulado", False),
            "rejeitados":     ("peças", "acumulado", False),
            "falhas":         ("falhas", "acumulado", False),
            "horas_operacao": ("a confirmar", "acumulado", False),
        }},
    "ESTEIRA_01": {
        "nome": "Esteira transportadora", "setor": "Movimentação", "ordem_fluxo": 4, "tabela": "prata_esteira",
        "campos": {
            "velocidade":     ("m/s", "instantaneo", True),
            "carga":          ("%", "instantaneo", True),
            "temperatura":    ("°C", "instantaneo", True),
            "vibracao":       ("mm/s", "instantaneo", True),
            "paradas":        ("paradas", "acumulado", True),
            "horas_operacao": ("a confirmar", "acumulado", True),
        }},
    "EMBALADORA_01": {
        "nome": "Embaladora", "setor": "Expedição", "ordem_fluxo": 5, "tabela": "prata_embaladora",
        "campos": {
            "velocidade":     ("a confirmar", "instantaneo", True),
            "temperatura":    ("°C", "instantaneo", True),
            "producao":       ("embalagens", "acumulado", True),
            "rejeitados":     ("embalagens", "acumulado", True),
            "falhas":         ("falhas", "acumulado", True),
            "horas_operacao": ("a confirmar", "acumulado", True),
        }},
    "COMPRESSOR_01": {
        "nome": "Compressor industrial", "setor": "Utilidades", "ordem_fluxo": 0, "tabela": "prata_compressor",
        "campos": {
            "pressao":        ("bar", "instantaneo", True),
            "temperatura":    ("°C", "instantaneo", True),
            "vibracao":       ("mm/s", "instantaneo", True),
            "consumo_kw":     ("kW", "instantaneo", True),
            "desgaste":       ("%", "nivel", True),
            "falhas":         ("falhas", "acumulado", True),
            "horas_operacao": ("h", "acumulado", True),
        }},
}

SEVERIDADE = {"Normal": 0, "Atenção": 1, "Crítico": 2, "Manutenção": 3}

# Limites do descritivo (Seção 08). (máquina, indicador, operador, limite, classificação, descrição)
# Uma leitura recebe a classificação MAIS SEVERA entre as regras que ela atinge.
# Depois da primeira execução, estas regras ficam na tabela config_limites e podem ser
# alteradas lá, sem mexer no código.
LIMITES = [
    ("TORNO_CNC_01", "temperatura", ">", 62,   "Atenção",    "Temperatura acima de 62 °C"),
    ("TORNO_CNC_01", "temperatura", ">", 67,   "Crítico",    "Temperatura acima de 67 °C"),
    ("TORNO_CNC_01", "vibracao",    ">", 2.8,  "Atenção",    "Vibração acima de 2,8 mm/s"),
    ("TORNO_CNC_01", "vibracao",    ">", 3.6,  "Crítico",    "Vibração acima de 3,6 mm/s"),
    ("TORNO_CNC_01", "rotacao",     "<", 2900, "Atenção",    "Rotação abaixo de 2.900 RPM"),
    ("TORNO_CNC_01", "rotacao",     "<", 2750, "Crítico",    "Rotação abaixo de 2.750 RPM"),
    ("TORNO_CNC_01", "desgaste",    ">", 60,   "Atenção",    "Desgaste acima de 60%"),
    ("TORNO_CNC_01", "desgaste",    ">", 85,   "Crítico",    "Desgaste acima de 85%"),
    ("TORNO_CNC_01", "desgaste",    ">", 90,   "Manutenção", "Desgaste acima de 90%"),

    ("PRENSA_HIDRAULICA_01", "pressao",     "<", 135, "Atenção", "Pressão abaixo de 135 bar"),
    ("PRENSA_HIDRAULICA_01", "pressao",     "<", 125, "Crítico", "Pressão abaixo de 125 bar"),
    ("PRENSA_HIDRAULICA_01", "temperatura", ">", 62,  "Atenção", "Temperatura acima de 62 °C"),
    ("PRENSA_HIDRAULICA_01", "temperatura", ">", 66,  "Crítico", "Temperatura acima de 66 °C"),
    ("PRENSA_HIDRAULICA_01", "vibracao",    ">", 2.5, "Atenção", "Vibração acima de 2,5 mm/s"),
    ("PRENSA_HIDRAULICA_01", "vibracao",    ">", 3.3, "Crítico", "Vibração acima de 3,3 mm/s"),

    ("ESTEIRA_01", "carga",       ">", 85,   "Atenção", "Carga acima de 85%"),
    ("ESTEIRA_01", "carga",       ">", 92,   "Crítico", "Carga acima de 92%"),
    ("ESTEIRA_01", "velocidade",  "<", 1.7,  "Atenção", "Velocidade abaixo de 1,7 m/s"),
    ("ESTEIRA_01", "velocidade",  "<", 1.55, "Crítico", "Velocidade abaixo de 1,55 m/s"),
    ("ESTEIRA_01", "temperatura", ">", 44,   "Atenção", "Temperatura acima de 44 °C"),
    ("ESTEIRA_01", "temperatura", ">", 47,   "Crítico", "Temperatura acima de 47 °C"),
    ("ESTEIRA_01", "vibracao",    ">", 3.0,  "Atenção", "Vibração acima de 3,0 mm/s"),
    ("ESTEIRA_01", "vibracao",    ">", 3.6,  "Crítico", "Vibração acima de 3,6 mm/s"),

    ("ROBO_SOLDAGEM_01", "temperatura", ">", 75,   "Atenção",    "Temperatura acima de 75 °C"),
    ("ROBO_SOLDAGEM_01", "temperatura", ">", 85,   "Crítico",    "Temperatura acima de 85 °C"),
    ("ROBO_SOLDAGEM_01", "corrente",    "<", 16.5, "Atenção",    "Corrente abaixo de 16,5 A"),
    ("ROBO_SOLDAGEM_01", "corrente",    ">", 20,   "Atenção",    "Corrente acima de 20 A"),
    ("ROBO_SOLDAGEM_01", "falhas",      ">", 2,    "Atenção",    "Falhas acumuladas entre 3 e 5"),
    ("ROBO_SOLDAGEM_01", "falhas",      ">", 5,    "Manutenção", "Falhas acumuladas acima de 5"),

    ("COMPRESSOR_01", "pressao",     "<", 7.5, "Atenção",    "Pressão abaixo de 7,5 bar"),
    ("COMPRESSOR_01", "pressao",     "<", 6.5, "Crítico",    "Pressão abaixo de 6,5 bar"),
    ("COMPRESSOR_01", "temperatura", ">", 75,  "Atenção",    "Temperatura acima de 75 °C"),
    ("COMPRESSOR_01", "temperatura", ">", 85,  "Crítico",    "Temperatura acima de 85 °C"),
    ("COMPRESSOR_01", "vibracao",    ">", 4.0, "Atenção",    "Vibração acima de 4,0 mm/s"),
    ("COMPRESSOR_01", "vibracao",    ">", 5.5, "Manutenção", "Vibração acima de 5,5 mm/s"),
    ("COMPRESSOR_01", "desgaste",    ">", 60,  "Atenção",    "Desgaste acima de 60%"),
    ("COMPRESSOR_01", "desgaste",    ">", 85,  "Crítico",    "Desgaste acima de 85%"),
    ("COMPRESSOR_01", "consumo_kw",  ">", 8.0, "Atenção",    "Consumo acima de 8,0 kW"),
    ("COMPRESSOR_01", "consumo_kw",  ">", 8.8, "Crítico",    "Consumo acima de 8,8 kW"),

    ("EMBALADORA_01", "velocidade",  ">",  35, "Atenção",    "Velocidade acima de 35"),
    ("EMBALADORA_01", "velocidade",  ">",  38, "Crítico",    "Velocidade acima de 38"),
    ("EMBALADORA_01", "velocidade",  "<",  25, "Atenção",    "Velocidade reduzida, abaixo de 25"),
    ("EMBALADORA_01", "temperatura", ">",  48, "Atenção",    "Temperatura acima de 48 °C"),
    ("EMBALADORA_01", "temperatura", ">",  50, "Crítico",    "Temperatura acima de 50 °C"),
    ("EMBALADORA_01", "falhas",      ">",  2,  "Atenção",    "Falhas acumuladas entre 3 e 4"),
    ("EMBALADORA_01", "falhas",      ">=", 5,  "Manutenção", "Falhas acumuladas: 5 ou mais"),
]

CABECALHO = ["tipo", "status", "maquina", "id", "timestamp"]   # campos do JSON que não são medidas
print(f"✔ Catálogo: {len(MAQUINAS)} máquinas, {len(LIMITES)} regras de limite")

In [ ]:
# ================================================================
# 4. CRIAÇÃO DAS TABELAS (estrutura documentada de cada uma)
# ================================================================
def acumulados(cfg):
    return [c for c, (_, nat, _) in cfg["campos"].items() if nat == "acumulado"]

ESQUEMAS = {
    # ---------------- BRONZE ----------------
    "bronze_leituras": ("Copia fiel das leituras do MySQL, sem alteracoes", [
        ("id_origem",   "BIGINT",    "id da leitura no MySQL"),
        ("maquina",     "STRING",    "identificador da maquina"),
        ("dados_json",  "STRING",    "JSON original enviado pela ESP32"),
        ("ts_origem",   "TIMESTAMP", "horario da leitura (UTC)"),
        ("ingestao_em", "TIMESTAMP", "quando o notebook importou a leitura"),
        ("lote",        "STRING",    "identificador da execucao que importou"),
    ]),
    # ---------------- PRATA ----------------
    "prata_quarentena": ("Leituras rejeitadas na validacao, com o motivo", [
        ("id_origem", "BIGINT", "id da leitura no MySQL"),
        ("maquina", "STRING", "maquina informada"),
        ("ts", "TIMESTAMP", "horario da leitura"),
        ("motivo", "STRING", "por que a leitura foi rejeitada"),
        ("dados_json", "STRING", "JSON original, como evidencia"),
        ("ingestao_em", "TIMESTAMP", "quando foi importada"),
    ]),
    "prata_leituras": ("Cabecalho de todas as leituras validas, de todas as maquinas", [
        ("id_origem", "BIGINT", "id da leitura no MySQL"),
        ("maquina", "STRING", "maquina"),
        ("tipo", "STRING", "tipo informado pela ESP32"),
        ("status", "STRING", "status informado pela ESP32"),
        ("ts", "TIMESTAMP", "horario da leitura (UTC)"),
        ("intervalo_s", "DOUBLE", "segundos desde a leitura anterior da mesma maquina"),
        ("reiniciou", "BOOLEAN", "contadores zeraram: provavel reinicio da ESP32"),
    ]),
    "prata_medidas": ("Formato longo: uma linha por medida de cada leitura", [
        ("id_origem", "BIGINT", "id da leitura no MySQL"),
        ("maquina", "STRING", "maquina"),
        ("ts", "TIMESTAMP", "horario da leitura (UTC)"),
        ("status", "STRING", "status informado"),
        ("indicador", "STRING", "nome do campo medido"),
        ("valor", "DOUBLE", "valor medido"),
        ("unidade", "STRING", "unidade do catalogo"),
        ("natureza", "STRING", "instantaneo, acumulado ou nivel"),
        ("no_catalogo", "BOOLEAN", "falso se o campo ainda nao esta no catalogo"),
    ]),
    # ---------------- OURO: dimensões e configuração ----------------
    "dim_maquina": ("Cadastro das maquinas", [
        ("maquina", "STRING", "identificador da maquina"),
        ("nome", "STRING", "nome amigavel"),
        ("setor", "STRING", "setor da fabrica"),
        ("ordem_fluxo", "INT", "posicao no fluxo produtivo (0 = utilidade)"),
        ("tabela_prata", "STRING", "tabela prata da maquina"),
        ("meta_producao_hora", "DOUBLE", "meta de producao por hora (preencher)"),
    ]),
    "dim_indicador": ("Cadastro dos indicadores de cada maquina", [
        ("maquina", "STRING", "maquina"),
        ("indicador", "STRING", "nome do campo"),
        ("unidade", "STRING", "unidade de medida"),
        ("natureza", "STRING", "instantaneo, acumulado ou nivel"),
        ("confirmado", "BOOLEAN", "campo ja observado nos dados reais"),
    ]),
    "config_limites": ("Regras de classificacao. Editavel sem alterar o codigo", [
        ("id_regra", "INT", "identificador da regra"),
        ("maquina", "STRING", "maquina"),
        ("indicador", "STRING", "indicador avaliado"),
        ("operador", "STRING", "comparacao: >, >=, < ou <="),
        ("limite", "DOUBLE", "valor de referencia"),
        ("classificacao", "STRING", "Atencao, Critico ou Manutencao"),
        ("severidade", "INT", "1 Atencao, 2 Critico, 3 Manutencao"),
        ("descricao", "STRING", "texto exibido como justificativa"),
        ("ativo", "BOOLEAN", "regra em uso"),
    ]),
    # ---------------- OURO: fatos ----------------
    "fato_classificacao": ("Classificacao de cada medida que possui regra de limite", [
        ("id_origem", "BIGINT", "id da leitura"),
        ("maquina", "STRING", "maquina"),
        ("ts", "TIMESTAMP", "horario da leitura"),
        ("status", "STRING", "status informado pela ESP32"),
        ("indicador", "STRING", "indicador"),
        ("valor", "DOUBLE", "valor observado"),
        ("classificacao", "STRING", "Normal, Atencao, Critico ou Manutencao"),
        ("severidade", "INT", "0 a 3"),
        ("operador", "STRING", "operador da regra atingida"),
        ("limite", "DOUBLE", "limite aplicado"),
        ("regra", "STRING", "descricao da regra atingida"),
    ]),
    "fato_ocorrencias": ("Alertas agrupados: um registro por episodio, nao por leitura", [
        ("id_ocorrencia", "STRING", "identificador estavel da ocorrencia"),
        ("tipo_ocorrencia", "STRING", "LIMITE, STATUS, COMUNICACAO ou REINICIO"),
        ("maquina", "STRING", "maquina"),
        ("indicador", "STRING", "indicador envolvido"),
        ("classificacao", "STRING", "pior classificacao do episodio"),
        ("severidade", "INT", "pior severidade do episodio"),
        ("valor_inicial", "DOUBLE", "valor no inicio do episodio"),
        ("valor_pior", "DOUBLE", "valor na leitura mais grave"),
        ("limite", "DOUBLE", "limite aplicado na leitura mais grave"),
        ("regra", "STRING", "regra aplicada"),
        ("inicio", "TIMESTAMP", "inicio do episodio"),
        ("fim", "TIMESTAMP", "quando voltou ao normal (vazio se ativa)"),
        ("ultima_leitura", "TIMESTAMP", "ultima leitura dentro do episodio"),
        ("duracao_s", "DOUBLE", "duracao em segundos"),
        ("qtd_leituras", "BIGINT", "leituras dentro do episodio"),
        ("status_ocorrencia", "STRING", "ATIVA ou RESOLVIDA"),
        ("ocorrencias_24h", "BIGINT", "ocorrencias iguais nas 24 h anteriores"),
        ("recorrente", "BOOLEAN", "3 ou mais ocorrencias iguais em 24 h"),
        ("justificativa", "STRING", "texto explicativo para o painel"),
    ]),
    "fato_paradas": ("Paradas por status PARADA e pelo contador de paradas", [
        ("maquina", "STRING", "maquina"),
        ("origem", "STRING", "STATUS ou CONTADOR"),
        ("inicio", "TIMESTAMP", "inicio da parada"),
        ("fim", "TIMESTAMP", "fim da parada (vazio se ativa)"),
        ("duracao_s", "DOUBLE", "duracao em segundos (so para origem STATUS)"),
        ("qtd_paradas", "DOUBLE", "paradas contadas"),
        ("ativa", "BOOLEAN", "parada em andamento"),
    ]),
    "fato_indicadores_periodo": ("Indicadores calculados por maquina e janela de tempo", [
        ("maquina", "STRING", "maquina"),
        ("inicio_janela", "TIMESTAMP", "inicio do periodo"),
        ("fim_janela", "TIMESTAMP", "fim do periodo"),
        ("janela_completa", "BOOLEAN", "periodo ja encerrado"),
        ("leituras", "BIGINT", "leituras no periodo"),
        ("tempo_observado_s", "DOUBLE", "tempo coberto por leituras"),
        ("tempo_operando_s", "DOUBLE", "tempo com status OPERANDO"),
        ("disponibilidade_pct", "DOUBLE", "tempo operando / tempo observado x 100"),
        ("producao", "DOUBLE", "producao do periodo (soma dos deltas)"),
        ("ciclos", "DOUBLE", "ciclos do periodo"),
        ("defeitos", "DOUBLE", "defeitos do periodo"),
        ("rejeitados", "DOUBLE", "rejeitados do periodo"),
        ("falhas", "DOUBLE", "falhas do periodo"),
        ("paradas", "DOUBLE", "paradas do periodo"),
        ("taxa_defeitos_pct", "DOUBLE", "defeitos / total processado x 100"),
        ("taxa_rejeicao_pct", "DOUBLE", "rejeitados / total processado x 100"),
        ("falhas_por_hora", "DOUBLE", "falhas / horas operando"),
        ("temperatura_media", "DOUBLE", "media da temperatura"),
        ("vibracao_max", "DOUBLE", "maior vibracao"),
        ("desgaste_final", "DOUBLE", "desgaste na ultima leitura do periodo"),
        ("consumo_kwh", "DOUBLE", "energia: potencia x tempo"),
        ("consumo_por_hora_kwh", "DOUBLE", "kWh por hora de operacao"),
        ("meta_periodo", "DOUBLE", "meta proporcional ao periodo"),
        ("pct_meta", "DOUBLE", "producao / meta x 100"),
        ("alerta_meta", "BOOLEAN", "producao abaixo de 70% da meta"),
        ("alerta_qualidade", "BOOLEAN", "defeitos ou rejeicao acima de 5%"),
    ]),
    "fato_estado_atual": ("Situacao atual de cada maquina, para os cartoes do painel", [
        ("maquina", "STRING", "maquina"),
        ("nome", "STRING", "nome amigavel"),
        ("ultima_leitura", "TIMESTAMP", "horario da ultima leitura"),
        ("segundos_sem_leitura", "DOUBLE", "segundos desde a ultima leitura"),
        ("sem_comunicacao", "BOOLEAN", "mais de 15 s sem leitura"),
        ("status_recebido", "STRING", "status enviado pela ESP32"),
        ("estado_geral", "STRING", "estado calculado pelo sistema"),
        ("severidade_max", "INT", "pior severidade na ultima leitura"),
        ("indicadores_alterados", "STRING", "indicadores fora do normal"),
        ("qtd_indicadores_alterados", "INT", "quantidade de indicadores fora do normal"),
        ("diagnostico_combinado", "BOOLEAN", "2 ou mais indicadores alterados ao mesmo tempo"),
        ("ocorrencias_ativas", "BIGINT", "ocorrencias ativas da maquina"),
        ("atualizado_em", "TIMESTAMP", "quando este estado foi calculado"),
    ]),
}

# Uma tabela Prata por máquina, gerada a partir do catálogo
for _maq, _cfg in MAQUINAS.items():
    _cols = [
        ("id_origem", "BIGINT", "id da leitura no MySQL"),
        ("maquina", "STRING", "maquina"),
        ("ts", "TIMESTAMP", "horario da leitura (UTC)"),
        ("status", "STRING", "status informado pela ESP32"),
        ("intervalo_s", "DOUBLE", "segundos desde a leitura anterior"),
    ]
    _cols += [(c, "DOUBLE", f"{c} em {u}, {nat}") for c, (u, nat, _) in _cfg["campos"].items()]
    _cols += [(f"delta_{c}", "DOUBLE", f"variacao de {c} desde a leitura anterior") for c in acumulados(_cfg)]
    _cols += [("reiniciou", "BOOLEAN", "contadores zeraram nesta leitura")]
    ESQUEMAS[_cfg["tabela"]] = (f"Leituras validas de {_maq}, com deltas dos contadores", _cols)


def ddl(nome):
    """Esquema da tabela no formato aceito pelo Spark: 'col1 TIPO, col2 TIPO'."""
    return ", ".join(f"{c} {t}" for c, t, _ in ESQUEMAS[nome][1])


def criar_tabela(nome):
    descricao, colunas = ESQUEMAS[nome]
    cols = ",\n  ".join(f"{c} {t} COMMENT '{d}'" for c, t, d in colunas)
    spark.sql(f"CREATE TABLE IF NOT EXISTS {nome} (\n  {cols}\n) USING {FORMATO} COMMENT '{descricao}'")


for _nome in ESQUEMAS:
    criar_tabela(_nome)
    print(f"✔ {_nome}")

In [ ]:
# ================================================================
# 5. FUNÇÕES AUXILIARES
# ================================================================
from pyspark.sql import functions as F, Window
from pyspark.sql.types import MapType, StringType


def gravar(df, nome):
    """Grava o DataFrame na tabela, na ordem e nos tipos documentados (substitui o conteúdo)."""
    colunas = ESQUEMAS[nome][1]
    df = df.select([
        (F.col(c) if c in df.columns else F.lit(None)).cast(t).alias(c) for c, t, _ in colunas
    ])
    df.write.format(FORMATO).mode("overwrite").option("overwriteSchema", "true").saveAsTable(nome)
    print(f"  ✔ {nome}: {spark.table(nome).count()} linhas")


def semear_se_vazia(nome, linhas):
    """Preenche tabelas de cadastro só na primeira vez, para não apagar edições feitas depois."""
    if spark.table(nome).limit(1).count() == 0:
        spark.createDataFrame(linhas, ddl(nome)).write.format(FORMATO).mode("append").saveAsTable(nome)
        print(f"  ✔ {nome}: preenchida com {len(linhas)} linhas")
    else:
        print(f"  • {nome}: já preenchida (mantida como está)")


def dividir(a, b, fator=1.0):
    """a / b x fator, ou vazio se b for zero ou vazio (evita divisão por zero)."""
    return F.when(b > 0, a / b * fator)


def episodios(df, chaves, estado):
    """
    Agrupa leituras consecutivas com o mesmo valor na coluna 'estado' (ignorando estado vazio).
    Devolve as linhas com: _grupo (número do episódio) e _prox_ts (horário da leitura seguinte).
    """
    w = Window.partitionBy(*chaves).orderBy("ts", "id_origem")
    e = F.coalesce(F.col(estado), F.lit("__normal__"))
    anterior = F.lag(e).over(w)
    return (df
        .withColumn("_prox_ts", F.lead("ts").over(w))
        .withColumn("_mudou", F.when(anterior.isNull() | (anterior != e), 1).otherwise(0))
        .withColumn("_grupo", F.sum("_mudou").over(w.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
        .filter(F.col(estado).isNotNull()))

In [ ]:
# ================================================================
# 6. BRONZE — importa do MySQL apenas as leituras novas
# ================================================================
def obter_senha():
    if KEY_VAULT_URL:
        return notebookutils.credentials.getSecret(KEY_VAULT_URL, KEY_VAULT_SEGREDO)
    if os.path.exists(ARQUIVO_SENHA):
        with open(ARQUIVO_SENHA, encoding="utf-8") as arquivo:
            return json.load(arquivo)["senha"]
    raise RuntimeError(
        "Senha do MySQL não configurada. Preencha KEY_VAULT_URL ou crie o arquivo "
        f"{ARQUIVO_SENHA} com o conteúdo {{\"senha\": \"...\"}}.")


def ler_origem(ultimo_id):
    """Busca no MySQL as leituras com id maior que o último já importado."""
    import mysql.connector
    conexao = mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORTA, database=MYSQL_BANCO,
        user=MYSQL_USUARIO, password=obter_senha(), connection_timeout=20)
    try:
        cursor = conexao.cursor()
        cursor.execute(
            f"SELECT id, maquina, dados, `timestamp` FROM {MYSQL_TABELA} WHERE id > %s ORDER BY id",
            (int(ultimo_id),))
        return cursor.fetchall()
    finally:
        conexao.close()


def normalizar_json(dados):
    if dados is None:
        return None
    if isinstance(dados, (bytes, bytearray)):
        return dados.decode("utf-8", "replace")
    if isinstance(dados, (dict, list)):
        return json.dumps(dados, ensure_ascii=False)
    return str(dados)


def normalizar_ts(ts):
    try:
        if ts is None:
            return None
        if isinstance(ts, str):
            ts = datetime.fromisoformat(ts.replace("Z", "+00:00"))
        if ts.tzinfo is None:
            ts = ts.replace(tzinfo=ZoneInfo(FUSO_DO_BANCO))
        return ts
    except (ValueError, TypeError):
        return None   # timestamp inválido: a leitura vai para a quarentena

In [ ]:
ultimo_id = spark.sql("SELECT COALESCE(MAX(id_origem), 0) AS m FROM bronze_leituras").first()["m"]
linhas = ler_origem(ultimo_id)

agora_utc = datetime.now(timezone.utc)
lote = agora_utc.strftime("%Y%m%dT%H%M%SZ")
registros = [(int(i), m, normalizar_json(d), normalizar_ts(t), agora_utc, lote) for i, m, d, t in linhas]

if registros:
    (spark.createDataFrame(registros, ddl("bronze_leituras"))
        .dropDuplicates(["id_origem"])
        .write.format(FORMATO).mode("append").saveAsTable("bronze_leituras"))

print(f"✔ Bronze: {len(registros)} leituras novas (último id anterior: {ultimo_id}). "
      f"Total: {spark.table('bronze_leituras').count()}")

In [ ]:
# ================================================================
# 7. PRATA — validação, quarentena, tabela longa e uma tabela por máquina
# ================================================================
lido = (spark.table("bronze_leituras").dropDuplicates(["id_origem"])
    .withColumn("j", F.from_json("dados_json", MapType(StringType(), StringType())))
    .withColumn("ts", F.col("ts_origem"))
    .withColumn("status", F.upper(F.trim(F.col("j")["status"])))
    .withColumn("tipo", F.col("j")["tipo"]))

# 7.1 Campos que não são números ou são negativos
problemas_campos = (lido.filter(F.col("j").isNotNull())
    .select("id_origem", F.explode("j").alias("campo", "texto"))
    .filter(~F.col("campo").isin(CABECALHO))
    .withColumn("valor", F.col("texto").cast("double"))
    .withColumn("motivo",
        F.when(F.col("texto").isNotNull() & F.col("valor").isNull(),
               F.concat(F.lit("campo '"), F.col("campo"), F.lit("' não numérico")))
         .when(F.col("valor") < 0,
               F.concat(F.lit("campo '"), F.col("campo"), F.lit("' negativo: "), F.col("texto"))))
    .filter(F.col("motivo").isNotNull())
    .groupBy("id_origem").agg(F.concat_ws("; ", F.collect_list("motivo")).alias("motivo_campos")))

# 7.2 Campos obrigatórios (os já confirmados no catálogo)
VAZIO = F.array().cast("array<string>")
obrigatorios = F.create_map(*[
    x for maq, cfg in MAQUINAS.items()
    for x in (F.lit(maq), F.array(*[F.lit(c) for c, (_, _, conf) in cfg["campos"].items() if conf]).cast("array<string>"))
])

validacao = (lido
    .join(problemas_campos, "id_origem", "left")
    .withColumn("faltando", F.array_except(
        F.coalesce(F.element_at(obrigatorios, F.coalesce(F.col("maquina"), F.lit(""))), VAZIO),
        F.coalesce(F.map_keys("j"), VAZIO)))
    .withColumn("motivo", F.concat_ws("; ",
        F.when(F.col("j").isNull(), F.lit("JSON inválido ou vazio")),
        F.when(F.col("maquina").isNull() | (F.trim("maquina") == ""), F.lit("máquina ausente")),
        F.when(F.col("maquina").isNotNull() & ~F.col("maquina").isin(list(MAQUINAS)),
               F.concat(F.lit("máquina desconhecida: "), F.col("maquina"))),
        F.when(F.col("j")["maquina"].isNotNull() & (F.col("j")["maquina"] != F.col("maquina")),
               F.lit("máquina do JSON diferente da coluna")),
        F.when(F.col("j").isNotNull() & F.col("status").isNull(), F.lit("status ausente")),
        F.when(F.col("ts").isNull(), F.lit("timestamp ausente ou inválido")),
        F.when(F.size("faltando") > 0,
               F.concat(F.lit("campo obrigatório ausente: "), F.array_join("faltando", ", "))),
        F.col("motivo_campos"))))

print("Gravando camada Prata...")
gravar(validacao.filter(F.col("motivo") != ""), "prata_quarentena")
validos = validacao.filter(F.col("motivo") == "").cache()

# 7.3 Tabela longa de medidas (todas as máquinas juntas)
catalogo_df = spark.createDataFrame(
    [(m, c, u, n, conf) for m, cfg in MAQUINAS.items() for c, (u, n, conf) in cfg["campos"].items()],
    ddl("dim_indicador"))

medidas = (validos
    .select("id_origem", "maquina", "ts", "status", F.explode("j").alias("indicador", "texto"))
    .filter(~F.col("indicador").isin(CABECALHO) & F.col("texto").isNotNull())
    .withColumn("valor", F.col("texto").cast("double"))
    .join(catalogo_df.drop("confirmado"), ["maquina", "indicador"], "left")
    .withColumn("no_catalogo", F.col("natureza").isNotNull()))
gravar(medidas, "prata_medidas")


# 7.4 Uma tabela por máquina, com deltas dos contadores e detecção de reinício
def montar_prata_maquina(maquina, cfg):
    campos = cfg["campos"]
    contadores = acumulados(cfg)
    w = Window.partitionBy("maquina").orderBy("ts", "id_origem")

    df = (validos.filter(F.col("maquina") == maquina)
        .select("id_origem", "maquina", "tipo", "ts", "status",
                *[F.col("j")[c].cast("double").alias(c) for c in campos])
        .withColumn("intervalo_s", F.col("ts").cast("double") - F.lag(F.col("ts").cast("double")).over(w)))

    sinais = []
    for c in contadores:
        df = df.withColumn(f"_ant_{c}", F.lag(c).over(w))
        sinais.append(F.coalesce(F.col(c) < F.col(f"_ant_{c}"), F.lit(False)))
    if "desgaste" in campos:   # desgaste também zera quando a ESP32 reinicia
        df = df.withColumn("_ant_desgaste", F.lag("desgaste").over(w))
        sinais.append(F.coalesce(F.col("desgaste") < F.col("_ant_desgaste") - 1, F.lit(False)))
    df = df.withColumn("reiniciou", reduce(or_, sinais) if sinais else F.lit(False))

    for c in contadores:
        anterior = F.col(f"_ant_{c}")
        df = df.withColumn(f"delta_{c}",
            F.when(F.col(c).isNull(), F.lit(None))
             .when(anterior.isNull(), F.lit(0.0))        # primeira leitura: sem base de comparação
             .when(F.col("reiniciou"), F.col(c))         # reinício: conta a partir do zero
             .otherwise(F.col(c) - anterior))
    return df


prata_por_maquina = {}
for maq, cfg in MAQUINAS.items():
    df = montar_prata_maquina(maq, cfg)
    gravar(df, cfg["tabela"])
    prata_por_maquina[maq] = spark.table(cfg["tabela"])

leituras_validas = reduce(lambda a, b: a.unionByName(b), [
    montar_prata_maquina(maq, cfg).select("id_origem", "maquina", "tipo", "status", "ts", "intervalo_s", "reiniciou")
    for maq, cfg in MAQUINAS.items()])
gravar(leituras_validas, "prata_leituras")

In [ ]:
# ================================================================
# 8. OURO — cadastros e configuração (preenchidos só na primeira vez)
# ================================================================
print("Cadastros da camada Ouro...")
semear_se_vazia("dim_maquina", [
    (m, cfg["nome"], cfg["setor"], cfg["ordem_fluxo"], cfg["tabela"], None) for m, cfg in MAQUINAS.items()])
semear_se_vazia("dim_indicador", [
    (m, c, u, n, conf) for m, cfg in MAQUINAS.items() for c, (u, n, conf) in cfg["campos"].items()])
semear_se_vazia("config_limites", [
    (i + 1, m, ind, op, float(lim), cl, SEVERIDADE[cl], desc, True)
    for i, (m, ind, op, lim, cl, desc) in enumerate(LIMITES)])

In [ ]:
# ================================================================
# 9. OURO — classificação de cada medida pelos limites
# ================================================================
limites = spark.table("config_limites").filter("ativo")
regra_atingida = (
    F.when(F.col("operador") == ">",  F.col("valor") >  F.col("limite"))
     .when(F.col("operador") == ">=", F.col("valor") >= F.col("limite"))
     .when(F.col("operador") == "<",  F.col("valor") <  F.col("limite"))
     .when(F.col("operador") == "<=", F.col("valor") <= F.col("limite"))
     .otherwise(F.lit(False)))

avaliacao = (spark.table("prata_medidas").filter(F.col("valor").isNotNull())
    .join(limites.select("maquina", "indicador", "operador", "limite", "classificacao", "severidade", "descricao"),
          ["maquina", "indicador"])
    .withColumn("peso", F.when(regra_atingida, F.col("severidade")).otherwise(F.lit(0))))

classificacao = (avaliacao
    .groupBy("id_origem", "maquina", "ts", "status", "indicador", "valor")
    .agg(F.max("peso").alias("severidade"),
         F.max_by(F.struct("classificacao", "operador", "limite", "descricao"), "peso").alias("r"))
    .select("id_origem", "maquina", "ts", "status", "indicador", "valor", "severidade",
            F.when(F.col("severidade") > 0, F.col("r.classificacao")).otherwise(F.lit("Normal")).alias("classificacao"),
            F.when(F.col("severidade") > 0, F.col("r.operador")).alias("operador"),
            F.when(F.col("severidade") > 0, F.col("r.limite")).alias("limite"),
            F.when(F.col("severidade") > 0, F.col("r.descricao")).alias("regra")))

print("Classificação...")
gravar(classificacao, "fato_classificacao")

In [ ]:
# ================================================================
# 10. OURO — ocorrências (alertas sem repetição) e paradas
# ================================================================
leituras = spark.table("prata_leituras")
agora = F.current_timestamp()
colunas_ocorrencia = [c for c, _, _ in ESQUEMAS["fato_ocorrencias"][1]]

# 10.1 LIMITE: leituras consecutivas fora do normal viram UMA ocorrência
ep = episodios(spark.table("fato_classificacao").withColumn(
    "_estado", F.when(F.col("severidade") > 0, F.lit("ALERTA"))), ["maquina", "indicador"], "_estado")
oc_limite = (ep.groupBy("maquina", "indicador", "_grupo")
    .agg(F.min("ts").alias("inicio"),
         F.max("ts").alias("ultima_leitura"),
         F.max_by("_prox_ts", "ts").alias("fim"),
         F.count("*").alias("qtd_leituras"),
         F.max("severidade").alias("severidade"),
         F.min_by("valor", "ts").alias("valor_inicial"),
         F.max_by(F.struct("classificacao", "valor", "limite", "regra"), F.struct("severidade", "ts")).alias("p"))
    .select("maquina", "indicador", "inicio", "ultima_leitura", "fim", "qtd_leituras", "severidade", "valor_inicial",
            F.lit("LIMITE").alias("tipo_ocorrencia"),
            F.col("p.classificacao").alias("classificacao"),
            F.col("p.valor").alias("valor_pior"),
            F.col("p.limite").alias("limite"),
            F.col("p.regra").alias("regra"))
    .withColumn("justificativa", F.concat(
        F.col("indicador"), F.lit(" = "), F.round("valor_pior", 2).cast("string"),
        F.lit(" → "), F.col("classificacao"), F.lit(" ("), F.col("regra"), F.lit(")"))))

# 10.2 STATUS: PARADA, MANUTENCAO ou qualquer status diferente de OPERANDO
ep = episodios(leituras.withColumn("_estado", F.when(F.col("status") != "OPERANDO", F.col("status"))),
               ["maquina"], "_estado")
oc_status = (ep.groupBy("maquina", "_grupo", "_estado")
    .agg(F.min("ts").alias("inicio"),
         F.max("ts").alias("ultima_leitura"),
         F.max_by("_prox_ts", "ts").alias("fim"),
         F.count("*").alias("qtd_leituras"))
    .withColumn("tipo_ocorrencia", F.lit("STATUS"))
    .withColumn("indicador", F.lit("status"))
    .withColumn("classificacao", F.when(F.col("_estado") == "MANUTENCAO", "Manutenção")
                                  .when(F.col("_estado") == "PARADA", "Parada").otherwise("Atenção"))
    .withColumn("severidade", F.when(F.col("_estado") == "MANUTENCAO", 3)
                               .when(F.col("_estado") == "PARADA", 2).otherwise(1))
    .withColumn("regra", F.concat(F.lit("Status recebido igual a "), F.col("_estado")))
    .withColumn("justificativa", F.concat(F.lit("Máquina informou status "), F.col("_estado"))))

# 10.3 COMUNICAÇÃO: intervalos de mais de 15 s entre leituras, e máquinas que pararam de enviar
oc_com_resolvidas = (leituras.filter(F.col("intervalo_s") > LIMITE_SEM_COMUNICACAO_S)
    .select("maquina",
            (F.col("ts").cast("double") - F.col("intervalo_s")).cast("timestamp").alias("inicio"),
            F.col("ts").alias("fim"),
            F.col("intervalo_s").alias("duracao_s")))
ultimas = (leituras.groupBy("maquina").agg(F.max("ts").alias("inicio"))
    .withColumn("duracao_s", agora.cast("double") - F.col("inicio").cast("double"))
    .filter(F.col("duracao_s") > LIMITE_SEM_COMUNICACAO_S)
    .withColumn("fim", F.lit(None).cast("timestamp")))
oc_comunicacao = (oc_com_resolvidas.unionByName(ultimas)
    .withColumn("ultima_leitura", F.col("inicio"))
    .withColumn("tipo_ocorrencia", F.lit("COMUNICACAO"))
    .withColumn("indicador", F.lit("comunicacao"))
    .withColumn("classificacao", F.lit("Crítico"))
    .withColumn("severidade", F.lit(2))
    .withColumn("qtd_leituras", F.lit(0))
    .withColumn("regra", F.lit(f"Sem nova leitura por mais de {LIMITE_SEM_COMUNICACAO_S} s"))
    .withColumn("justificativa", F.concat(
        F.lit("Sem comunicação por "), F.round("duracao_s").cast("long").cast("string"), F.lit(" s; última leitura às "),
        F.date_format("inicio", "dd/MM HH:mm:ss"), F.lit(" UTC"))))

# 10.4 REINÍCIO: contadores zerados (ESP32 reiniciou)
oc_reinicio = (leituras.filter("reiniciou")
    .select("maquina", F.col("ts").alias("inicio"), F.col("ts").alias("fim"), F.col("ts").alias("ultima_leitura"))
    .withColumn("tipo_ocorrencia", F.lit("REINICIO"))
    .withColumn("indicador", F.lit("contadores"))
    .withColumn("classificacao", F.lit("Atenção"))
    .withColumn("severidade", F.lit(1))
    .withColumn("qtd_leituras", F.lit(1))
    .withColumn("regra", F.lit("Contador menor que na leitura anterior"))
    .withColumn("justificativa", F.lit("Contadores zerados: provável reinicialização da ESP32")))

# 10.5 Junta tudo, calcula duração, status e recorrência
ocorrencias = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True),
                     [oc_limite, oc_status, oc_comunicacao, oc_reinicio])
w_rec = (Window.partitionBy("tipo_ocorrencia", "maquina", "indicador")
         .orderBy(F.col("inicio").cast("long")).rangeBetween(-86400, 0))
ocorrencias = (ocorrencias
    .withColumn("duracao_s", F.coalesce(
        F.col("duracao_s"),
        F.coalesce(F.col("fim"), F.col("ultima_leitura")).cast("double") - F.col("inicio").cast("double")))
    .withColumn("status_ocorrencia", F.when(
        F.col("fim").isNull() & (F.col("tipo_ocorrencia") != "REINICIO"), "ATIVA").otherwise("RESOLVIDA"))
    .withColumn("ocorrencias_24h", F.count("*").over(w_rec))
    .withColumn("recorrente", F.col("ocorrencias_24h") >= 3)
    .withColumn("id_ocorrencia", F.substring(F.sha2(F.concat_ws(
        "|", "tipo_ocorrencia", "maquina", "indicador", F.col("inicio").cast("string")), 256), 1, 16)))

print("Ocorrências e paradas...")
gravar(ocorrencias, "fato_ocorrencias")

# 10.6 Paradas: episódios de status PARADA + aumentos do contador de paradas
paradas_status = (spark.table("fato_ocorrencias")
    .filter((F.col("tipo_ocorrencia") == "STATUS") & (F.col("classificacao") == "Parada"))
    .select("maquina", F.lit("STATUS").alias("origem"), "inicio", "fim", "duracao_s",
            F.lit(1.0).alias("qtd_paradas"), (F.col("status_ocorrencia") == "ATIVA").alias("ativa")))
partes = [paradas_status]
for maq, cfg in MAQUINAS.items():
    if "paradas" in cfg["campos"]:
        partes.append(prata_por_maquina[maq].filter(F.col("delta_paradas") > 0)
            .select("maquina", F.lit("CONTADOR").alias("origem"), F.col("ts").alias("inicio"),
                    F.lit(None).cast("timestamp").alias("fim"), F.lit(None).cast("double").alias("duracao_s"),
                    F.col("delta_paradas").alias("qtd_paradas"), F.lit(False).alias("ativa")))
gravar(reduce(lambda a, b: a.unionByName(b), partes), "fato_paradas")

In [ ]:
# ================================================================
# 11. OURO — indicadores por período (produção, qualidade, disponibilidade...)
# ================================================================
partes = []
for maq, cfg in MAQUINAS.items():
    df = prata_por_maquina[maq]
    tempo = F.least(F.coalesce(F.col("intervalo_s"), F.lit(0.0)), F.lit(float(INTERVALO_MAXIMO_S)))
    aggs = [F.count("*").alias("leituras"),
            F.sum(tempo).alias("tempo_observado_s"),
            F.sum(F.when(F.col("status") == "OPERANDO", tempo).otherwise(F.lit(0.0))).alias("tempo_operando_s")]
    for c in ["producao", "ciclos", "defeitos", "rejeitados", "falhas", "paradas"]:
        if f"delta_{c}" in df.columns:
            aggs.append(F.sum(f"delta_{c}").alias(c))
    if "temperatura" in df.columns:
        aggs.append(F.avg("temperatura").alias("temperatura_media"))
    if "vibracao" in df.columns:
        aggs.append(F.max("vibracao").alias("vibracao_max"))
    if "desgaste" in df.columns:
        aggs.append(F.max_by("desgaste", "ts").alias("desgaste_final"))
    if "consumo_kw" in df.columns:   # potência (kW) x tempo (h) = energia (kWh)
        aggs.append(F.sum(F.col("consumo_kw") * tempo / 3600).alias("consumo_kwh"))
    partes.append(df.groupBy("maquina", F.window("ts", JANELA_INDICADORES).alias("janela")).agg(*aggs))

horas_operando = F.col("tempo_operando_s") / 3600
total_processado = F.coalesce(F.col("ciclos"), F.col("producao"))   # prensa e robô: ciclos
metas = spark.table("dim_maquina").select("maquina", "meta_producao_hora")

indicadores = (reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True), partes)
    .join(metas, "maquina", "left")
    .withColumn("inicio_janela", F.col("janela.start"))
    .withColumn("fim_janela", F.col("janela.end"))
    .withColumn("janela_completa", F.col("fim_janela") <= agora)
    .withColumn("disponibilidade_pct", dividir(F.col("tempo_operando_s"), F.col("tempo_observado_s"), 100.0))
    .withColumn("taxa_defeitos_pct", dividir(F.col("defeitos"), total_processado, 100.0))
    .withColumn("taxa_rejeicao_pct", dividir(F.col("rejeitados"), total_processado, 100.0))
    .withColumn("falhas_por_hora", dividir(F.col("falhas"), horas_operando))
    .withColumn("consumo_por_hora_kwh", dividir(F.col("consumo_kwh"), horas_operando))
    .withColumn("meta_periodo", F.col("meta_producao_hora") *
                (F.col("fim_janela").cast("double") - F.col("inicio_janela").cast("double")) / 3600)
    .withColumn("pct_meta", dividir(F.col("producao"), F.col("meta_periodo"), 100.0))
    .withColumn("alerta_meta", F.col("janela_completa") & (F.col("pct_meta") < 70))
    .withColumn("alerta_qualidade",
                F.coalesce(F.greatest("taxa_defeitos_pct", "taxa_rejeicao_pct") > 5, F.lit(False))))

print("Indicadores por período...")
gravar(indicadores, "fato_indicadores_periodo")

In [ ]:
# ================================================================
# 12. OURO — estado atual de cada máquina (cartões do painel)
# ================================================================
w_ultima = Window.partitionBy("maquina").orderBy(F.col("ts").desc(), F.col("id_origem").desc())
ultima = leituras.withColumn("_n", F.row_number().over(w_ultima)).filter("_n = 1")

alteracoes = (spark.table("fato_classificacao").join(ultima.select("id_origem"), "id_origem")
    .groupBy("maquina")
    .agg(F.max("severidade").alias("severidade_max"),
         F.concat_ws(", ", F.collect_list(F.when(F.col("severidade") > 0, F.concat(
             F.col("indicador"), F.lit(" ("), F.col("classificacao"), F.lit(")"))))).alias("indicadores_alterados"),
         F.sum((F.col("severidade") > 0).cast("int")).alias("qtd_indicadores_alterados")))

ativas = (spark.table("fato_ocorrencias").filter(F.col("status_ocorrencia") == "ATIVA")
    .groupBy("maquina").agg(F.count("*").alias("ocorrencias_ativas")))

estado = (spark.table("dim_maquina").select("maquina", "nome")
    .join(ultima.select("maquina", F.col("ts").alias("ultima_leitura"), F.col("status").alias("status_recebido")),
          "maquina", "left")
    .join(alteracoes, "maquina", "left")
    .join(ativas, "maquina", "left")
    .fillna({"severidade_max": 0, "qtd_indicadores_alterados": 0, "ocorrencias_ativas": 0, "indicadores_alterados": ""})
    .withColumn("segundos_sem_leitura", agora.cast("double") - F.col("ultima_leitura").cast("double"))
    .withColumn("sem_comunicacao", F.coalesce(F.col("segundos_sem_leitura") > LIMITE_SEM_COMUNICACAO_S, F.lit(True)))
    .withColumn("estado_geral",
        F.when(F.col("ultima_leitura").isNull(), "Sem dados")
         .when(F.col("sem_comunicacao"), "Sem comunicação")
         .when((F.col("status_recebido") == "MANUTENCAO") | (F.col("severidade_max") == 3), "Manutenção")
         .when(F.col("status_recebido") == "PARADA", "Parada")
         .when(F.col("severidade_max") == 2, "Crítico")
         .when(F.col("severidade_max") == 1, "Atenção")
         .otherwise("Normal"))
    .withColumn("diagnostico_combinado", F.col("qtd_indicadores_alterados") >= 2)
    .withColumn("atualizado_em", agora))

print("Estado atual...")
gravar(estado, "fato_estado_atual")
validos.unpersist()

In [ ]:
# ================================================================
# 13. VERIFICAÇÃO FINAL DO LAKEHOUSE
# ================================================================
print(f"Tabelas no Lakehouse '{spark.catalog.currentDatabase()}':\n")
for nome in ESQUEMAS:
    print(f"  {nome:<26} {spark.table(nome).count():>8} linhas")

print("\nEstado atual das máquinas:")
spark.table("fato_estado_atual").select(
    "maquina", "estado_geral", "status_recebido", "indicadores_alterados", "ocorrencias_ativas"
).orderBy("maquina").show(truncate=False)

fora = (spark.table("prata_medidas").filter(~F.col("no_catalogo"))
        .groupBy("maquina", "indicador").count())
if fora.limit(1).count() > 0:
    print("⚠ Campos recebidos que ainda não estão no catálogo (atualize MAQUINAS na célula 3):")
    fora.show(truncate=False)

q = spark.table("prata_quarentena").count()
if q:
    print(f"⚠ {q} leituras na quarentena. Motivos:")
    spark.table("prata_quarentena").groupBy("motivo").count().show(truncate=False)

## Próximos passos
- **Power BI:** no Lakehouse, abra o *SQL analytics endpoint* ou crie um **modelo semântico** com as tabelas
  `dim_*` e `fato_*`. Relacione tudo pela coluna `maquina` com `dim_maquina`.
- **Metas de produção:** preencha `meta_producao_hora` na tabela `dim_maquina` para ativar o alerta de meta.
- **Limites:** para ajustar uma regra, altere a linha correspondente em `config_limites`.
  A próxima execução já usa o valor novo.
- **Prensa e robô:** quando essas máquinas começarem a enviar dados, confira os nomes dos campos.
  A verificação final avisa se aparecer algum campo fora do catálogo.